# 03 · Collect frames from your own NBA clips for labelling

Public datasets only go so far: on a real NBA clip the models found about 6 of 10 players and 5 of
18 court landmarks per frame. Labelled frames from the footage the pipeline actually runs on close
that gap fastest, and the cheapest ones to label are the frames the current models get wrong.

This notebook runs both models over your clips, scores every sampled frame by how much they
struggled (missing players, no ball or several balls, unsure boxes, too few court landmarks,
players overlapping in the paint, motion blur) and exports the hardest frames, spread out in time,
with the models' predictions as pre-labels, so in Roboflow you only correct them.

**Test clips stay separate.** Clips listed in `TEST_CLIPS` are never mined for training. Frames
from them are exported at an even spacing (a fair sample, not the hardest frames) into separate
folders for a separate Roboflow project: once corrected, they are the `own_test` and
`own_court_test` sources that notebooks 01 and 02 score on and never train on.

| Folder in `training_data/to_label/` | Correct in Roboflow, then download (YOLO format) to `training_data/` |
|---|---|
| `detector/` | `own_labelled/` |
| `court/` | `own_labelled_court/` |
| `detector_test/` | `own_test/` |
| `court_test/` | `own_court_test/` |
| `negative_candidates/` | look through; move frames with nothing in play to `negatives/` |

**Which clips:** variety beats volume. 15 to 20 games, 3 to 5 minutes each, many teams (including
dark-vs-dark and white-vs-light jerseys), arenas, broadcasters and quarters; half-court sets with
a crowded paint, fast breaks with pans and blur. Keep 2 to 3 whole games as test clips.

In [ ]:
# Runtime > Change runtime type > GPU before running (A100 or L4 if you have them; T4 works, slower).
!nvidia-smi -L
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"   # everything that must survive a disconnect lives here
for d in ["models", "runs", "reports", "training_data", "input_videos", "output"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"   # to test a pull request, put its branch name here, then Runtime > Restart session > Run all
!rm -rf /content/courtvision && git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
!pip -q install -e "/content/courtvision[models]"
import sys
sys.path.insert(0, "/content/courtvision")   # importable now, without restarting the session

## Settings

`STRIDE` samples one frame in N (every 10th frame of 30 fps video is 3 per second). `MIN_GAP`
keeps chosen frames apart so one hard moment does not fill the budget. Boxes between `LOW_CONF`
and `KEEP_CONF` count as unsure; pre-labels use `KEEP_CONF`.

In [ ]:
import glob
from pathlib import Path
CLIPS = sorted(glob.glob(f"{DRIVE_ROOT}/input_videos/*.mp4"))
TEST_CLIPS = []                 # file names of whole clips to keep for testing, e.g. ["lakers_spurs_q2.mp4"]
FRAMES_PER_CLIP = 40            # training frames mined per clip
TEST_FRAMES_PER_CLIP = 30       # evenly spaced frames per test clip
STRIDE, MIN_GAP = 10, 45        # frames
LOW_CONF, KEEP_CONF, KP_CONF = 0.10, 0.25, 0.5
OUT = f"{DRIVE_ROOT}/training_data/to_label"
train_clips = [c for c in CLIPS if Path(c).name not in TEST_CLIPS]
test_clips = [c for c in CLIPS if Path(c).name in TEST_CLIPS]
print(len(train_clips), "clips to mine:", [Path(c).name for c in train_clips])
print(len(test_clips), "test clips:", [Path(c).name for c in test_clips])

## Score the frames of the training clips

In [ ]:
import cv2, numpy as np
from ultralytics import YOLO
from courtvision.detect import best_court_instance
from courtvision.training.mining import FrameStats, overlap_pairs, pick_frames, sharpness
from courtvision.training.sources import DETECT_CLASSES, class_map

det = YOLO(f"{DRIVE_ROOT}/models/player_detector.pt")
court_model = YOLO(f"{DRIVE_ROOT}/models/court_keypoints.pt")
to_ours = class_map([det.names[i] for i in sorted(det.names)], DETECT_CLASSES)   # model id -> our id
PLAYER, BALL = DETECT_CLASSES.index("Player"), DETECT_CLASSES.index("Ball")

def predict(frame, conf):
    """(boxes xyxy, our class ids, confidences) for our four classes."""
    b = det.predict(frame, imgsz=1280, conf=conf, verbose=False)[0].boxes
    cls = np.array([to_ours.get(int(c), -1) for c in b.cls.cpu().numpy()], int)
    keep = cls >= 0
    return b.xyxy.cpu().numpy()[keep], cls[keep], b.conf.cpu().numpy()[keep]

def keypoints(frame):
    return best_court_instance(court_model.predict(frame, imgsz=640, conf=0.25, verbose=False)[0])

def score_clip(path):
    cap, stats, f = cv2.VideoCapture(path), [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if f % STRIDE == 0:
            xyxy, cls, conf = predict(frame, LOW_CONF)
            sure = conf >= KEEP_CONF
            players = sure & (cls == PLAYER)
            gray = cv2.cvtColor(cv2.resize(frame, (960, 540)), cv2.COLOR_BGR2GRAY)  # fixed size: comparable blur
            stats.append(FrameStats(frame=f, players=int(players.sum()), balls=int((sure & (cls == BALL)).sum()),
                                    uncertain=int((~sure).sum()), keypoints=int((keypoints(frame)[:, 2] >= KP_CONF).sum()),
                                    overlaps=overlap_pairs(xyxy[players]), sharpness=sharpness(gray)))
        f += 1
    cap.release()
    return stats

all_stats = {c: score_clip(c) for c in train_clips}
for c, s in all_stats.items():
    print(f"{Path(c).name:32s} frames={len(s):4d}  players={np.mean([x.players for x in s]):.1f}"
          f"  no ball={np.mean([x.balls == 0 for x in s]):.0%}  keypoints={np.mean([x.keypoints for x in s]):.1f}"
          f"  crowded={np.mean([x.overlaps > 0 for x in s]):.0%}  blurred={np.mean([x.sharpness < 100 for x in s]):.0%}")

## Export with pre-labels

Detector pre-labels are the model's boxes in the four training classes. Court pre-labels are the
geometry-checked mapping when the model's keypoints pass the check (every landmark in the frame
placed, including ones the model missed); otherwise just the keypoints it was sure of. Frames
where the court model sees nothing at all are usually replays, close-ups or crowd shots and go to
`negative_candidates/`.

In [ ]:
import os, shutil, yaml
from courtvision.court import NBA
from courtvision.training.labels import box_rows, court_row_from_keypoints, pose_row

def read_frame(path, idx):
    cap = cv2.VideoCapture(path); cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read(); cap.release()
    return frame if ok else None

def export(clip, frames, det_dir, court_dir):
    """Write frames of one clip with detector and court pre-labels. Returns (detector, court, negatives)."""
    n = [0, 0, 0]
    for idx in frames:
        frame = read_frame(clip, idx)
        if frame is None:
            continue
        name, size = f"{Path(clip).stem}_f{idx:06d}", (frame.shape[1], frame.shape[0])
        kps = keypoints(frame)
        if (kps[:, 2] >= KP_CONF).sum() == 0:
            cv2.imwrite(f"{OUT}/negative_candidates/{name}.jpg", frame); n[2] += 1
            continue
        xyxy, cls, _ = predict(frame, KEEP_CONF)
        cv2.imwrite(f"{det_dir}/images/{name}.jpg", frame)
        Path(f"{det_dir}/labels/{name}.txt").write_text("\n".join(box_rows(xyxy, cls, size)) + "\n"); n[0] += 1
        row, _ = court_row_from_keypoints(kps, NBA, size)
        row = row or pose_row(kps[:, :2], kps[:, 2] >= KP_CONF, size)
        cv2.imwrite(f"{court_dir}/images/{name}.jpg", frame)
        Path(f"{court_dir}/labels/{name}.txt").write_text((row or "") + "\n"); n[1] += 1
    return n

for sub in ["detector", "court", "detector_test", "court_test"]:
    for kind in ["images", "labels"]:
        os.makedirs(f"{OUT}/{sub}/{kind}", exist_ok=True)
os.makedirs(f"{OUT}/negative_candidates", exist_ok=True)

for clip, stats in all_stats.items():
    print(Path(clip).name, export(clip, pick_frames(stats, FRAMES_PER_CLIP, min_gap=MIN_GAP), f"{OUT}/detector", f"{OUT}/court"))
for clip in test_clips:
    n = int(cv2.VideoCapture(clip).get(cv2.CAP_PROP_FRAME_COUNT))
    even = np.linspace(0, n - 1, TEST_FRAMES_PER_CLIP).astype(int).tolist()
    print(Path(clip).name, "(test)", export(clip, even, f"{OUT}/detector_test", f"{OUT}/court_test"))

for sub in ["detector", "detector_test"]:
    yaml.safe_dump({"names": dict(enumerate(DETECT_CLASSES))}, open(f"{OUT}/{sub}/data.yaml", "w"))
for sub in ["court", "court_test"]:
    yaml.safe_dump({"names": {0: "court"}, "kpt_shape": [18, 3]}, open(f"{OUT}/{sub}/data.yaml", "w"))

In [ ]:
# Zip each folder for Roboflow's upload page (it reads the YOLO labels as pre-labels). A keypoint
# project needs its 18 landmarks defined first, in the order of notebook 02.
for sub in ["detector", "court", "detector_test", "court_test"]:
    print(shutil.make_archive(f"{OUT}/{sub}", "zip", f"{OUT}/{sub}"))